# Laboratorio 6 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

# Task 1

1. Un desarrollador junior de su equipo sugiere: "Para detectar con mayor precisión las texturas de las
hojas enfermas, deberíamos construir una red secuencial clásica (tipo VGG) pero de 150 capas. Más
profundo siempre es mejor". Como líder técnico, explíquele argumentativamente por qué esta red
fracasará estrepitosamente en el entrenamiento (mencionando el fenómeno de degradación y el
desvanecimiento del gradiente). Luego, justifique cómo la adición estructural de las conexiones
residuales (𝐹(𝑥) + 𝑥) de ResNet rescata el proyecto, haciendo viable entrenar redes ultra-profundas
sin colapsar.

R//

La propuesta de construir una red secuencial clásica tipo VGG con 150 capas parte de una intuición incompleta. Es cierto que VGG mostró que aumentar la profundidad puede mejorar la capacidad de representación, pero existe un límite práctico y matemático. En redes muy profundas aparecen dos problemas centrales. El primero es el desvanecimiento del gradiente, donde durante la retropropagación los gradientes se vuelven cada vez más pequeños al atravesar muchas capas, por lo que las capas tempranas casi dejan de aprender. El segundo es la degradación de la precisión, que significa que al seguir agregando capas el error de entrenamiento aumenta en vez de disminuir. El material de clase lo resume de forma directa al indicar que una mayor profundidad puede producir mayor error de entrenamiento y que esto no debe confundirse con sobreajuste.

En una arquitectura puramente secuencial, cada capa debe aprender una transformación completa sobre la salida anterior. En teoría esto parece más expresivo, pero en la práctica vuelve muy inestable la optimización. Si una parte de la red adicional no aporta valor, el modelo no tiene un mecanismo simple para comportarse como identidad y dejar pasar la información sin distorsionarla. Por eso una VGG extrema de 150 capas tendería a entrenar lentamente, estancarse o incluso rendir peor que una versión mucho más corta, aun teniendo más capacidad nominal.

ResNet rescata el proyecto porque cambia la pregunta que aprende cada bloque. En lugar de obligar a la red a aproximar directamente una transformación completa H(x), el bloque residual aprende solo el residuo F(x), de modo que la salida final se expresa como H(x) = F(x) + x. Esta suma con la identidad no es un detalle cosmético, sino una modificación estructural profunda. En la retropropagación aparece un término adicional igual a 1 en la derivada con respecto a la entrada del bloque, lo cual crea una ruta directa para el flujo del gradiente y reduce el desvanecimiento. En otras palabras, aunque la rama convolucional aprenda poco o se vuelva difícil de optimizar, la señal todavía puede propagarse por el atajo de identidad.

Desde la perspectiva del liderazgo técnico, la conclusión es clara. No conviene apostar el proyecto AgriTech a una VGG de 150 capas solo por ser más profunda. Esa decisión elevaría el riesgo de entrenamiento fallido, pérdida de tiempo de GPU y retrasos de entrega. En cambio, una ResNet convierte la profundidad en una ventaja utilizable, porque preserva el flujo de información y de gradientes, reduce el colapso de optimización y vuelve factible entrenar modelos ultra profundos para capturar texturas finas en hojas enfermas sin destruir la estabilidad del proceso.


2. Las enfermedades en las hojas de mango son visualmente heterogéneas: La Antracnosis se
presenta como puntos negros diminutos, mientras que el Moho Polvoriento cubre áreas enormes de
la hoja. Analice cómo la topología en paralelo del módulo Inception (usando filtros 3x3 y 5x5
simultáneamente) es ideal para este problema biológico en particular. Además, desde una
perspectiva de costos de infraestructura (uso de GPUs en AWS o Google Cloud), explique cómo la
inserción estratégica de convoluciones de 1x1 evita la explosión de la dimensionalidad y salva el
presupuesto mensual de la startup.

R//

El problema biológico de las hojas de mango exige una arquitectura que vea patrones en múltiples escalas espaciales al mismo tiempo. La Antracnosis puede manifestarse como puntos negros diminutos, lo que requiere filtros pequeños capaces de capturar detalles locales y texturas finas. En cambio, el Moho Polvoriento cubre regiones más amplias de la hoja, por lo que conviene usar filtros mayores que integren contexto espacial más global. El material de clase justamente explica que los filtros pequeños como 3x3 capturan detalles finos, mientras que filtros más grandes como 5x5 o 7x7 capturan contexto global.

Aquí es donde el módulo Inception resulta especialmente adecuado. Su topología en paralelo evita obligar al diseñador a escoger un único tamaño de filtro antes de entrenar. En lugar de eso, procesa la misma entrada por varias ramas simultáneas, por ejemplo con 3x3 y 5x5, y luego concatena los mapas de características obtenidos. Así, la red aprende de manera conjunta rasgos pequeños y patrones extensos. Para un dominio tan heterogéneo como la fitopatología foliar, esto es una ventaja decisiva, porque una sola imagen puede contener tanto lesiones puntuales como zonas infectadas de gran cobertura. El módulo Inception fue precisamente concebido como una extracción a múltiples escalas espaciales.

Sin embargo, esa riqueza multiescala tiene un peligro operativo. Si cada rama procesa todos los canales de entrada con convoluciones costosas y luego se concatenan todas las salidas, la dimensionalidad puede crecer rápidamente. Esto incrementa parámetros, memoria intermedia y FLOPs, lo cual se traduce en tiempos de entrenamiento más altos y facturas más caras en GPUs de AWS o Google Cloud. El material lo describe como una explosión combinatoria de dimensiones y advierte que la concatenación puede generar mapas inmanejables.

La solución elegante es insertar convoluciones de 1x1 antes de las ramas 3x3 y 5x5. Matemáticamente, estas capas actúan como proyectores lineales sobre la dimensión de canales. No cambian la resolución espacial, pero sí reducen C, es decir, la cantidad de canales que entrará a las convoluciones posteriores. Como el costo computacional de la convolución crece con N, C, W y H, reducir canales antes de aplicar filtros más grandes disminuye de manera importante las operaciones requeridas. Por eso el material destaca que las convoluciones 1x1 permiten reducción preventiva de dimensionalidad y mantienen alta expresividad con bajo costo.

Traducido a decisiones de negocio, esto significa que Inception no solo mejora la sensibilidad del modelo ante síntomas visuales muy distintos, sino que también protege el presupuesto mensual de la startup. Menos FLOPs implican menor tiempo por época, menor uso de memoria y posibilidad de entrenar lotes razonables sin subir a instancias más costosas. En un entorno startup, donde cada hora de GPU cuenta, usar 1x1 antes de 3x3 y 5x5 no es una optimización menor, sino una medida financiera y técnica al mismo tiempo. Por ello, para AgriTech, Inception ofrece un balance muy conveniente entre diversidad espacial y eficiencia computacional.

3. El cliente final (el agricultor) usará la aplicación en un teléfono Android de gama baja en medio del
campo, sin conexión a internet. Sabemos que MobileNet logra esta eficiencia gracias a la Depthwise
Separable Convolution. Describa brevemente cómo esta convolución divide el trabajo (filtrado
espacial vs. combinación de canales). Sin embargo, en ingeniería no hay soluciones mágicas, todo tiene un precio: ¿Qué capacidad matemática o expresividad de la red se está sacrificando al separar
estos dos pasos en comparación con una convolución estándar? Analice críticamente por qué, como
director del proyecto, usted acepta pagar ese "costo" en este escenario comercial.

R//



# Task 2